# 🕵️ Dia 76 — Comunicando um caso de fraude ao IC3 ou a um centro nacional antifraude

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 76* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-76.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 3/nov. O Dr. Hélio quer saber se o material "aguenta uma pergunta de defesa". Rafa cola na parede um décimo cartão, **V4 — Seu Antônio**, e puxa fios até as chaves de L-03 e L-05. O cartão W1, sozinho desde o Dia 68, ganha um fio.

Neste notebook você vai:

1. **Gerar e conferir** o pacote do IC3 (caso do original) e o pacote do BO (Seu Antônio).
2. **Jogar o quiz "qual porta primeiro?"**.
3. **Pôr R7 e V4 no grafo** e calcular os agrupamentos por vínculo forte.
4. **Gerar o relatório de uma página** para a Marta.
5. **Reler o `minha-linha.md` do Dia 1** e escrever a revisão de fim de fase.

> 🚫 Nada daqui é enviado a portal real: nem IC3, nem delegacia online. Comunicação falsa de crime é crime (CP, art. 340).

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo aqui é fictício (Operação Isca).

## 1. Os dados do Dia 75

Os mesmos números da linha do tempo do Dia 75, embutidos aqui para o notebook funcionar sozinho.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

UTC, CHI, SP = ZoneInfo("UTC"), ZoneInfo("America/Chicago"), ZoneInfo("America/Sao_Paulo")

# Caso do original (fictício): pagamentos do E03, em UTC
PAG_FICT = [("2026-01-22T15:03:11Z", 1000, "TFICT-A1", "0xFICT01"), ("2026-02-03T16:40:55Z", 5000, "TFICT-A1", "0xFICT02"),
            ("2026-02-10T17:02:09Z", 4000, "TFICT-A1", "0xFICT03"), ("2026-02-18T15:20:47Z", 6000, "TFICT-B7", "0xFICT04"),
            ("2026-02-26T19:11:30Z", 3000, "TFICT-B7", "0xFICT05")]
# ➕ Seu Antônio (fictício): Pix do E04, em UTC
PIX_ISCA = [("2026-10-26T11:14:22Z", 4800.00, "L-03", "PIXFICT-L03", "E2E-FICT-0001"),
            ("2026-10-26T11:31:05Z", 4950.00, "L-05", "PIXFICT-L05", "E2E-FICT-0002")]

def local(iso_utc, fuso):
    """'2026-01-22T15:03:11Z' → '2026-01-22 09:03 (America/Chicago)'."""
    t = datetime.strptime(iso_utc, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=UTC)
    return t.astimezone(fuso).strftime("%Y-%m-%d %H:%M") + f" ({fuso.key})"

for p in PAG_FICT:
    print(p[0], "→", local(p[0], CHI))
for p in PIX_ISCA:
    print(p[0], "→", local(p[0], SP))

## 2. O pacote do IC3 (caso do original) — e o Checkpoint em código

Cinco seções, uma por área do formulário do IC3. Depois, uma função confere o pacote contra a lista do FAQ do IC3: o que faltar é "uma lacuna para a qual um órgão teria de voltar a pedir".

In [ ]:
PACOTE_IC3 = {
    "vitima": "Lena Doe, 100 Example St, +1 312 555 0101 (FICTÍCIO)",
    "transacoes": [
        {"data_utc": p[0], "data_local": local(p[0], CHI), "valor": f"{p[1]} USDT (~US$ {p[1]})",
         "meio": "transferência de criptomoeda a partir de conta de corretora", "recebedor": p[2], "tx_hash": p[3]}
        for p in PAG_FICT],
    "sujeito": [
        ("carteira", "TFICT-A1", "E03"), ("carteira", "TFICT-B7", "E03"),
        ("telefone", "+1 202 555 0147", "E01"), ("domínio", "aurum-desk.example", "E02/E04"),
        ("nome usado", "Lena (destinatária do 'número errado')", "E01"),
        ("telefone — SUJEITO SEPARADO", "+1 646 555 0108 'Chainsafe Recovery'; nenhuma evidência o liga ao operador original", "E05"),
    ],
    "narrativa": ("Em janeiro recebi uma mensagem de número errado perguntando do jantar de sexta. Fiquei conversando com a pessoa, "
                  "que me apresentou um aplicativo de ouro, aurum-desk.example. Entre 22 de janeiro e 26 de fevereiro mandei cinco "
                  "pagamentos em USDT, 19 mil no total. Quando quis sacar, o aplicativo pediu 15% de imposto. Em 9 de março um tal "
                  "de Chainsafe Recovery ligou oferecendo recuperar o dinheiro por 1.500 dólares."),
    "outras": "Corretora contatada em 2026-03-10, referência FICT-REF-0001; denúncia também feita na plataforma de mensagens.",
}

def conferir_pacote(pacote, campos_obrigatorios):
    """Devolve a lista de lacunas: campo ausente, vazio, ou transação sem recebedor."""
    lacunas = [c for c in campos_obrigatorios if not pacote.get(c)]
    for i, t in enumerate(pacote.get("transacoes", []), start=1):
        if not t.get("recebedor"):
            lacunas.append(f"transação {i} sem conta/endereço recebedor")
    if len(pacote.get("narrativa", "").split()) > 250:
        lacunas.append("narrativa com mais de 250 palavras")
    return lacunas

FAQ_IC3 = ["vitima", "transacoes", "sujeito", "narrativa"]
lac = conferir_pacote(PACOTE_IC3, FAQ_IC3)
print("Pacote IC3:", "completo ✅" if not lac else f"lacunas: {lac}")
print(f"Transações: {len(PACOTE_IC3['transacoes'])} | narrativa: {len(PACOTE_IC3['narrativa'].split())} palavras")

## 3. ➕ O pacote do BO do Seu Antônio

As seções que um registro de ocorrência por estelionato costuma pedir (a ordem varia por estado). Repare na seção extra que o IC3 não tem: **local do fato** (a competência é do domicílio da vítima, CPP art. 70, § 4º).

In [ ]:
#@title Pacote do BO (fictício) { display-mode: "form" }
vitima = "Antônio da Silva Exemplo, 71 anos, CPF fictício 000.000.000-00, +55 00 00000-0099"  #@param {type:"string"}
local_do_fato = "Domicílio da vítima, cidade-sede da Serra Azul (MG) — FICTÍCIO"  #@param {type:"string"}
narrativa = "No domingo à noite recebi um áudio do Cláudio, meu gerente, dizendo que ia me ligar cedo por causa do Pix bloqueado. Na segunda ele ligou, disse que tinha gente tentando tirar dinheiro da minha conta e que eu tinha que passar para uma conta de segurança. Fiquei no telefone e fiz dois Pix. À tarde liguei para a agência e descobri que não era ele. Na quarta um advogado ligou dizendo que recuperava o Pix por oitocentos reais."  #@param {type:"string"}

PACOTE_BO = {
    "vitima": vitima,
    "local_do_fato": local_do_fato,
    "transacoes": [
        {"data_utc": p[0], "data_local": local(p[0], SP), "valor": f"R$ {p[1]:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
         "meio": "Pix", "recebedor": f"conta do laranja {p[2]} (chave {p[3]}, fictícia)", "id_transacao": p[4]}
        for p in PIX_ISCA],
    "sujeito": [
        ("telefone (áudio e ligação)", "+55 00 00000-0021", "E02/E03"),
        ("contas recebedoras", "L-03, L-05 (identificação completa só para autoridade e banco)", "E04"),
        ("FATO SEPARADO — tentativa de golpe de recuperação", "+55 00 00000-0029 'Dr. Silva'; sem vínculo afirmado", "E05"),
    ],
    "narrativa": narrativa,
    "providencias": "Pedido de devolução aberto na cooperativa (protocolo MED-FICT-0001, E06); número denunciado no WhatsApp.",
    "evidencias": "E01–E07 preservados com SHA-256 (registro de evidências do Dia 75); originais disponíveis.",
}

CAMPOS_BO = ["vitima", "local_do_fato", "transacoes", "sujeito", "narrativa", "providencias", "evidencias"]
lac = conferir_pacote(PACOTE_BO, CAMPOS_BO)
print("Pacote BO:", "completo ✅" if not lac else f"lacunas: {lac}")

from pathlib import Path
import hashlib
linhas = ["# Pacote para boletim de ocorrência — FICTÍCIO, NÃO ENVIAR", ""]
for chave in CAMPOS_BO:
    linhas.append(f"## {chave.replace('_', ' ').capitalize()}")
    valor = PACOTE_BO[chave]
    if isinstance(valor, list):
        for item in valor:
            linhas.append("- " + (" | ".join(f"{k}: {v}" for k, v in item.items()) if isinstance(item, dict) else " — ".join(item)))
    else:
        linhas.append(valor)
    linhas.append("")
Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia-76-pacote-bo.md")
saida.write_text("\n".join(linhas), encoding="utf-8")
print("\n".join(linhas))
print("💾", saida, "| SHA-256:", hashlib.sha256(saida.read_bytes()).hexdigest())

## 4. 🎮 Quiz: qual porta primeiro?

Para cada situação, qual é o **primeiro** passo? `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

PORTAS = {"a": "Banco/instituição (bloqueio, devolução, MED)", "b": "Boletim de ocorrência (Polícia Civil)",
          "c": "Denúncia na plataforma", "d": "Cartão do Dia 70 (SaferNet, Disque 100, 190)"}

QUIZ = [
    ("🇧🇷 Seu Antônio acabou de perceber, às 13h, que os dois Pix da manhã eram golpe.", "a",
     "Mesmo dia, primeiro o dinheiro: a instituição abre o MED. O BO vem logo em seguida."),
    ("🇺🇸 Vítima do caso FICT-075, um dia depois do último pagamento em cripto.", "a",
     "O original: banco ou corretora primeiro, para devolução ou bloqueio."),
    ("🇧🇷 No meio da entrevista, aparece uma menor de idade sendo aliciada pelo mesmo perfil.", "d",
     "Antes de tudo: checagem de segurança e o cartão do Dia 70."),
    ("🇧🇷 O dinheiro já foi bloqueado pelo banco; o perfil falso 'Serra Azul Atendimento' segue ativo no WhatsApp.", "c",
     "A conta ainda faz vítimas: denúncia na plataforma para remoção (o BO segue em paralelo)."),
    ("🇧🇷 O banco abriu o pedido de devolução e pediu à vítima o número do registro policial.", "b",
     "Agora o BO, com o pacote do Dia 76."),
    ("🇧🇷 Você quer 'testar' se a delegacia online aceita o seu pacote de treino.", "nenhuma",
     "Nunca. Comunicação falsa de crime é crime (CP, art. 340). Responda qualquer letra: esta não tem porta certa."),
]

acertos = 0
for i, (sit, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {sit}")
    for k, v in PORTAS.items():
        print(f"   {k}) {v}")
    if MODO_JOGO:
        r = ""
        for _ in range(5):
            r = input("   Primeira porta (a/b/c/d): ").strip().lower()
            if r in PORTAS:
                break
        ok = (r == certa) or certa == "nenhuma"
        acertos += ok
        print("   ✅" if ok else f"   ❌ era {certa}")
    print("   →", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 5. O grafo: R7 (original) e V4 (Operação Isca)

Primeiro, o caso do original: os seis relatos R1–R6 do Dia 68 do original mais o R7. Depois, o grafo da Operação Isca do Dia 68 deste repositório (baixado do GitHub ou embutido) mais o V4.

Os agrupamentos saem de *union-find* (Inovação OSINT do dia), **só com arestas fortes** (peso 3).

In [ ]:
import csv, io, urllib.request

def agrupamentos(arestas, so_peso=3):
    """Une as pontas de cada aresta com o peso pedido; devolve grupos de nós."""
    pai = {}
    def raiz(x):
        pai.setdefault(x, x)
        while pai[x] != x:
            pai[x] = pai[pai[x]]                      # compressão de caminho
            x = pai[x]
        return x
    for a, b, rotulo, peso in arestas:
        raiz(a); raiz(b)                              # registra os dois nós, mesmo sem unir
        if peso == so_peso:
            pai[raiz(a)] = raiz(b)
    grupos = {}
    for no in pai:
        grupos.setdefault(raiz(no), set()).add(no)
    return list(grupos.values())

def casos(grupo, prefixos):
    return sorted(n for n in grupo if n.startswith(prefixos) and not n.startswith(("PH", "P0", "WA", "WB")))

# ---------- Caso do original (Day 68 + R7) ----------
AR_ORIG = [("R1","P0147","texted from",3),("R1","WA1","paid to",3),("R1","D1","app domain",3),("R1","N1","claimed employer",2),
           ("R1","PH1","used phrase",1),("R2","P0183","texted from",3),("R2","WA1","paid to",3),("R2","D2","app domain",3),
           ("R3","P0120","texted from",3),("R3","WB7","paid to",3),("R3","D3","app domain",3),("R3","N1","claimed employer",2),
           ("R4","P0120","recruiter phone",3),("R4","D4","email domain",3),("R5","P0199","recruiter phone",3),
           ("R5","PH1","used phrase",1),("R6","P0162","texted from",3)]
R7 = [("R7","WA1","paid to",3),("R7","WB7","paid to",3),("R7","P0147","first contact from",3),
      ("R7","AD","platform domain",3),("R7","P0108","recovery call from",3)]

for nome, ar in (("antes", AR_ORIG), ("com R7", AR_ORIG + R7)):
    g = [casos(x, ("R",)) for x in agrupamentos(ar)]
    print(f"Original {nome}: {[x for x in g if x]}")
vizinhos_0108 = {a for a, b, *_ in AR_ORIG + R7 if b == "P0108"}
print("P0108 toca:", vizinhos_0108, "→ sujeito separado ✅" if vizinhos_0108 == {"R7"} else "⚠️")

In [ ]:
# ---------- ➕ Operação Isca (Dia 68 deste repo + V4) ----------
URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-68/arestas.csv"
# Plano B: o arestas.csv do Dia 68 embutido (Source, Target, Label, Weight)
ARESTAS_68 = [
    ('V1', 'D1', 'dominio da isca', 3),
    ('V1', 'T01', 'SMS de', 3),
    ('V1', 'K01', 'Pix para', 3),
    ('V2', 'D1', 'dominio da isca', 3),
    ('V2', 'T02', 'SMS de', 3),
    ('V2', 'K01', 'Pix para', 3),
    ('V2', 'K02', 'Pix para', 3),
    ('V3', 'D1', 'dominio da isca', 3),
    ('V3', 'T04', 'SMS de', 3),
    ('V3', 'K03', 'Pix para', 3),
    ('E1', 'T11', 'telefone do recrutador', 3),
    ('E1', 'K03', 'conta da recrutada usada para repasse', 3),
    ('E1', 'N1', 'empresa alegada', 2),
    ('E2', 'T12', 'mensagem de', 3),
    ('E2', 'K02', 'taxa paga para', 3),
    ('E2', 'D2', 'dominio do app', 3),
    ('E2', 'N1', 'empresa alegada', 2),
    ('E2', 'PH1', 'usou a frase', 1),
    ('E3', 'T13', 'telefone do recrutador', 3),
    ('E3', 'KX9', 'taxa paga para', 3),
    ('E3', 'PH1', 'usou a frase', 1),
    ('E4', 'T11', 'telefone do recrutador', 3),
    ('E4', 'N1', 'empresa alegada', 2),
    ('BECB', 'D3', 'dominio do remetente', 3),
    ('BECB', 'K04', 'nova chave informada', 3),
    ('W1', 'T15', 'mensagem de', 3),
    ('W1', 'K05', 'Pix para', 3),
]

try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        AR_ISCA = [(l["Source"], l["Target"], l["Label"], int(l["Weight"]))
                   for l in csv.DictReader(io.StringIO(r.read().decode("utf-8")))]
    print("✅ arestas.csv do Dia 68 baixado.")
except Exception as erro:
    AR_ISCA = ARESTAS_68
    print(f"⚠️ Sem acesso ({type(erro).__name__}); usando o grafo embutido.")

V4 = [("V4", "K03", "Pix para", 3), ("V4", "K05", "Pix para", 3),
      ("V4", "T21", "áudio e ligação de", 3), ("V4", "T29", "ligação de recuperação de", 3)]

PREF = ("V", "E", "BECB", "W1")
antes = sorted([casos(x, PREF) for x in agrupamentos(AR_ISCA) if casos(x, PREF)], key=len, reverse=True)
depois = sorted([casos(x, PREF) for x in agrupamentos(AR_ISCA + V4) if casos(x, PREF)], key=len, reverse=True)
print("Agrupamentos fortes ANTES do V4: ", antes)
print("Agrupamentos fortes DEPOIS do V4:", depois)

principal = depois[0]
print("\nV4 no agrupamento principal?", "V4" in principal)
print("W1 deixou de estar isolado?", "W1" in principal)
print("T29 toca outro caso?", {a for a, b, *_ in AR_ISCA + V4 if b == "T29"} - {"V4"} or "não → sujeito separado ✅")
print("\nLeitura (palavra certa: 'provável'): a chave de L-05 liga o WhatsApp clonado da Coopvale (W1)")
print("ao mesmo agrupamento do 'Pix bloqueado' e do falso emprego, via V4.")

## 6. 📝 O relatório de uma página para a Marta

Na ordem da Parte 2 do original: achado com confiança → evidências → vínculos → ações → riscos e lacunas.

In [ ]:
#@title Relatório do caso V4 { display-mode: "form" }
achado = "Um golpe de falsa central com voz atribuída ao gerente da agência levou R$ 9.750,00 do cooperado em dois Pix para duas contas de laranjas (L-03 e L-05) em 26/10/2026 (confirmado pelo extrato, E04)."  #@param {type:"string"}
risco_principal = "Não sabemos quem gravou o áudio nem se a voz foi clonada ou imitada; E07 prova só onde o gerente estava."  #@param {type:"string"}

rel = f"""# Relatório do caso V4 — Operação Isca (FICTÍCIO)

**Achado.** {achado}

**Evidências.** E04 extrato Pix (SHA-256 no registro do Dia 75); E03 histórico de chamadas; E02 áudio exportado; E06 protocolo MED-FICT-0001; E07 declaração do RH.

**Vínculos.** Com só vínculos fortes, o V4 entra no agrupamento principal {'(confirmado no grafo)' if 'V4' in principal else ''} pela chave de L-03, compartilhada com V3 (Pix bloqueado) e E1 (falso emprego). A chave de L-05 liga, pela primeira vez, o W1 (WhatsApp clonado, Coopvale) a esse agrupamento. Leitura: provavelmente a mesma operação usa as mesmas contas de laranjas em três modalidades de golpe. O número da ligação de recuperação (+55 00 00000-0029) não se liga a nenhum outro caso e não é atribuído à operação.

**Ações.** Tomadas: pedido de devolução pela cooperativa (MED, no mesmo dia); número denunciado no WhatsApp; cooperado avisado de que a oferta de 'recuperação' é outro golpe. Recomendadas: BO na Polícia Civil de MG com o pacote anexo (a vítima tem mais de 70 anos: ação penal não depende de representação, CP art. 171, § 5º); alerta aos cooperados sobre falsa central com voz conhecida; compartilhar com a Coopvale o vínculo de L-05.

**Riscos e lacunas.** {risco_principal} O segundo Pix (11:31 UTC) é posterior ao fim registrado da ligação (11:28 UTC): checar o relógio do print. Nada liga, por enquanto, o +55 00 00000-0021 aos remetentes do SMS de 5/out; uma correspondência no grafo mudaria isso.
"""
saida = Path("notas/dia-76-relatorio-marta.md")
saida.write_text(rel, encoding="utf-8")
print(rel)
print("💾", saida, "| SHA-256:", hashlib.sha256(saida.read_bytes()).hexdigest())

## 7. ✍️ A linha, revisitada

O fim do Dia 76 do original: releia o que você escreveu no Dia 1 sobre o que faria você parar, e escreva três ou quatro frases sobre se a resposta mudou e sobre o momento desta fase em que a linha pareceu mais perto.

Se o `notas/minha-linha.md` do Dia 1 estiver nesta pasta (suba pelo ícone de pasta do Colab), ele aparece abaixo.

In [ ]:
#@title Revisão de fim da Fase 5 { display-mode: "form" }
mudou = "Minha linha não mudou de lugar, mas ficou mais concreta: agora sei que a tentação mais forte não veio do kit, veio da vontade de resolver o caso do Seu Antônio."  #@param {type:"string"}
momento_mais_perto = "Quando o número do 'Dr. Silva' encaixava tão bem na história que eu quis colocá-lo no grafo sem evidência."  #@param {type:"string"}

linha_dia1 = Path("notas/minha-linha.md")
if linha_dia1.is_file():
    print("📜 O que você escreveu no Dia 1:\n")
    print(linha_dia1.read_text(encoding="utf-8"))
else:
    print("(notas/minha-linha.md do Dia 1 não está aqui; suba o arquivo para relê-lo.)")

revisao = f"# Minha linha — revisão no fim da Fase 5\n\n{mudou}\n\nO momento em que a linha pareceu mais perto: {momento_mais_perto}\n"
Path("notas/dia-76-minha-linha-revisao.md").write_text(revisao, encoding="utf-8")
print("\n💾 notas/dia-76-minha-linha-revisao.md")
try:
    from google.colab import files
    for arq in ("notas/dia-76-pacote-bo.md", "notas/dia-76-relatorio-marta.md", "notas/dia-76-minha-linha-revisao.md"):
        files.download(arq)
except ImportError:
    print("(Fora do Colab: tudo ficou em notas/)")

## ✅ Antes de fechar o Dia 76 (e a Fase 5)

- [ ] Pacote IC3 completo contra o FAQ + quadro comparativo Canadá/Reino Unido
- [ ] Pacote do BO do Seu Antônio, com o "Dr. Silva" como fato separado
- [ ] Relatório com a primeira frase e a confiança; recuperação não atribuída
- [ ] R7 ligado por TFICT-A1, TFICT-B7 e +1 202 555 0147; P0108 isolado
- [ ] V4 ligado por L-03; W1 deixa de estar isolado por L-05; T29 isolado
- [ ] Revisão do `minha-linha.md` guardada com o quadro de progresso
- [ ] **Nada** enviado a portal real

**Próximo (Dia 77, Fase 6):** a Serra Azul tem uma assistente virtual na nuvem, a "Lia". Alguém andou conversando com ela de um jeito estranho.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>